In [1]:
import nltk

nltk.download('punkt')
nltk.download('punkt tab')
nltk.download('stop wordls')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SJ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Error loading punkt tab: Package 'punkt tab' not found in
[nltk_data]     index
[nltk_data] Error loading stop wordls: Package 'stop wordls' not found
[nltk_data]     in index


False

사전학습된 임베딩 사용하지 않는 경우

In [2]:
sentences = [
    'nice great best amazing',  # 긍정 문장 예시
    'stop lies',                # 부정/비판 문장 예시
    'pitiful nerd',             # 부정 문장 예시
    'excellent work',           # 긍정 문장 예시
    'supreme quality',          # 긍정 문장 예시
    'bad',                      # 부정 문장 예시
    'highly respectable'        # 긍정 문장 예시
]                               # 분류 모델에 넣을 입력 문장 리스트(list[str])
labels = [1, 0, 0, 1, 1, 0, 1]  # 각 문장에 대한 이진 라벨(1=긍정, 0=부정)

In [3]:
# NLTK 토크나이저로 토큰화
from nltk.tokenize import word_tokenize

tokenized_sentences = [word_tokenize(sent) for sent in sentences]
tokenized_sentences

[['nice', 'great', 'best', 'amazing'],
 ['stop', 'lies'],
 ['pitiful', 'nerd'],
 ['excellent', 'work'],
 ['supreme', 'quality'],
 ['bad'],
 ['highly', 'respectable']]

In [4]:
# 단어 사전 생성 + 정수 인코딩
from collections import Counter

tokens = [token for sent in tokenized_sentences for token in sent] # 문장 리스트를 1차원으로 평탄화
word_counts = Counter(tokens)  # 전체 토큰의 등장 갯수
print(word_counts)

word_to_index = {word: index + 2 for index, word in enumerate(tokens)} # 토큰을 순서대로 인덱싱(인덱스)
word_to_index['<PAD>'] = 0 # 패딩 토큰 추가
word_to_index['UNK'] = 1   # OOV 토큰 추가
word_to_index = dict(sorted(word_to_index.items(), key = lambda x:x[1])) # 딕셔너리 정렬(인덱스 순)
print(word_to_index)

vocab_size = len(word_to_index)  # 특수토큰 포함 전체 어휘수
vocab_size

Counter({'nice': 1, 'great': 1, 'best': 1, 'amazing': 1, 'stop': 1, 'lies': 1, 'pitiful': 1, 'nerd': 1, 'excellent': 1, 'work': 1, 'supreme': 1, 'quality': 1, 'bad': 1, 'highly': 1, 'respectable': 1})
{'<PAD>': 0, 'UNK': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


17

In [5]:
# 토큰화된 문장 리스트를 받아 단어 -> 인덱스 사전으로 정수 시퀀스 생성하는 함수
def texts_to_sequences(sentences, word_to_index):
    sequences = []

    for sent in sentences: # 문장 단위 순회
        sequence = []

        for token in sent: # 토큰 단위 순회
            if token in word_to_index: # 사전에 있는 단어면
                sequence.append(word_to_index[token]) # 해당 단어의 값(ID) 추가
            else:
                sequence.append(word_to_index['<UNK>']) # OOV 토큰 추가

        sequences.append(sequence)
    
    return sequences

sequences = texts_to_sequences(tokenized_sentences, word_to_index)
sequences

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]

In [6]:
# 패딩 추가
import numpy as np

# 서로 다른 길이의 정수 시퀀스를 0(<PAD>)으로 채워 (문장수, maxlen) 형태로 맞추는 함수
def pad_sequences(sequences, maxlen):
    # (문장수 x maxlen) 크기의 0 패딩 생성
    padded_sequences = np.zeros((len(sequences), maxlen), dtype=int)

    for index, seq in enumerate(sequences):
        # index번째 행에서 0번 위치부터 len(seq)-1 위치까지를 seq의 처음부터 maxlen개까지 사용
        padded_sequences[index, :len(seq)] = seq[:maxlen]  # 앞에서부터 시퀀스 채움. 시퀀스가 길면 maxlen으로 자름
    return padded_sequences

padded_sequences = pad_sequences(sequences, maxlen = 4)
padded_sequences   # (문장 수, maxlen = 4) 형태의 정수 배열 

array([[ 2,  3,  4,  5],
       [ 6,  7,  0,  0],
       [ 8,  9,  0,  0],
       [10, 11,  0,  0],
       [12, 13,  0,  0],
       [14,  0,  0,  0],
       [15, 16,  0,  0]])

In [7]:
padded_sequences.shape # 문장 수, 고정길이 4

(7, 4)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 정수 시퀀스를 임베딩 -> RNN -> 선형층으로 처리해서 이진분류 LOGIT(1개) 출력하는 모델
class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,   # 단어 사전 크기
            embedding_dim = embedding_dim, # 임베딩 차원
            padding_idx = 0                # 패딩 0 인덱스는 업데이트 하지않음
        )
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first = True) # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환 (차후 BCEWithLogitsLoss 등 사용해서 확률값 변환해야함)
        self.out = nn.Linear(hidden_size, 1) 
    def forward(self, x):
        embedded = self.embedding(x)  # (batch, seq_len) -> (batch, seq_len, embedding_dim)
        out, h_n = self.rnn(embedded) # out:logit값 , h_n : (num_layers*directions, batch, hidden_size)
        out = self.out(h_n.squeeze(0)) # (batch, hidden_size) -> (batch,1)
        return out # logit 값

embedding_dim = 100 # 단어 벡터 차원 크기
model = SimpleNet(vocab_size, embedding_dim, hidden_size = 16)
model


SimpleNet(
  (embedding): Embedding(17, 100, padding_idx=0)
  (rnn): RNN(100, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [9]:
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [10]:
from torchinfo import summary

summary(model)

Layer (type:depth-idx)                   Param #
SimpleNet                                --
├─Embedding: 1-1                         1,700
├─RNN: 1-2                               1,888
├─Linear: 1-3                            17
Total params: 3,605
Trainable params: 3,605
Non-trainable params: 0

In [11]:
from torchinfo import summary

summary(model) # 모델의 레이어 구성/ 파라미터 수 요약정보

Layer (type:depth-idx)                   Param #
SimpleNet                                --
├─Embedding: 1-1                         1,700
├─RNN: 1-2                               1,888
├─Linear: 1-3                            17
Total params: 3,605
Trainable params: 3,605
Non-trainable params: 0

In [12]:
# 임베딩 가중치 확인
import pandas as pd

wv = model.embedding.weight.data # 임베딩 층의 가중치 행렬(단어 ID x 임베딩 차원)
print(wv.shape)  # (vocab_size, embedding_dim)

vocab = word_to_index.keys()    # 인덱스만 가져옴
pd.DataFrame(wv, index = vocab) # 인덱스 추가해서 데이터프레임으로 확인

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
UNK,0.016543,-0.293688,2.341132,-0.683298,-0.729739,-0.847741,0.424179,-0.719046,0.872072,0.686273,...,0.430634,-1.260317,-0.312759,-2.366752,0.156519,0.734862,1.407544,-0.107634,2.132063,1.788215
nice,-0.011697,0.562725,-0.879182,0.379658,-1.993820,0.389143,0.192187,-2.393990,-1.000169,3.247478,...,1.451375,0.139634,-2.229912,0.828824,1.368820,0.348792,-0.059410,0.086586,1.233812,-1.251441
great,-1.223762,0.558075,1.653839,1.635572,-1.061443,-0.348871,0.153473,1.126654,-1.951426,-0.160757,...,-0.332608,2.816414,0.440935,-0.378236,-0.160168,0.481721,-2.982040,-0.972937,0.078157,1.295296
best,-1.066077,-0.905623,0.190133,0.871408,-0.521184,1.208572,0.533777,-0.269307,0.741648,-2.180998,...,-0.377955,-0.160091,0.214471,-1.422616,1.063691,1.593238,-0.821688,-1.114486,0.504567,-0.804334
amazing,-0.613923,0.123712,2.656893,-0.863564,0.204375,0.676356,0.558449,-0.560131,0.003214,-0.857133,...,-0.015215,-0.061745,-1.302603,-0.372922,0.190415,-0.807146,2.100880,2.534232,0.424746,0.909070
stop,-1.339416,-0.643212,-0.820782,0.962593,1.183102,-0.832370,-1.361693,-0.159076,-0.588645,0.762309,...,1.131431,-0.938533,-1.401069,-0.328791,-2.031504,-0.313537,0.037703,-0.387527,0.527846,1.689151
lies,0.049794,1.171721,0.960463,0.807093,-0.071718,0.464778,1.098570,-2.235607,-0.869260,-2.394771,...,-1.029270,-0.244684,1.103261,0.219618,-0.872202,1.302615,-1.302554,-1.317302,0.194595,1.141803
pitiful,0.888695,-1.124358,-2.077770,-0.220706,-0.325942,0.122925,-1.323478,-0.169258,-1.603880,-0.389279,...,1.076683,-1.779896,0.755942,1.459493,0.509381,-0.005742,0.031732,-0.356466,-0.032614,0.829337
nerd,0.971779,0.481957,-1.176232,0.020541,0.538829,2.464346,1.106139,-1.149529,0.847667,-1.028527,...,0.484349,-0.400662,1.141540,1.215269,0.522882,0.016653,-0.769177,2.587379,-0.407511,-0.689621


In [13]:
# 모델학습 준비 : 텐서 변환, DataLoader 구성, 손실함수/최적화함수 설정
X = torch.tensor(padded_sequences, dtype = torch.long)     # 입력 시퀀스는 LongTensor로 변환(Embedding 입력)
y = torch.tensor(labels, dtype = torch.float).unsqueeze(1) # 라벨은 FloatTensor로 변환. (N, ) -> (N,1)

dataset = TensorDataset(X,y)
dataloader = DataLoader(dataset, batch_size = 2, shuffle = True)

criterion = nn.BCEWithLogitsLoss() # 시그모이드를 포함한 손실함수
optimizer = optim.Adam(model.parameters(), lr = 0.005)

In [14]:
for epoch in range(20):
    epoch_loss = 0

    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()    # 이전 기울기 초기화
        output = model(x_batch)  # 순전파
        loss = criterion(output, y_batch) # 손실 계산
        loss.backward()          # 역전파 : 기울기 계산
        optimizer.step()         # 파라미터 업데이트

        epoch_loss += loss.item() # 미니배치 손실은 python float형태로 누적

    print(f"Epoch {epoch+1} Loss : {epoch_loss / len(dataloader)}")


Epoch 1 Loss : 0.6765948235988617
Epoch 2 Loss : 0.49768683686852455
Epoch 3 Loss : 0.44730299711227417
Epoch 4 Loss : 0.38211264461278915
Epoch 5 Loss : 0.32353922352194786
Epoch 6 Loss : 0.2631520442664623
Epoch 7 Loss : 0.2095935419201851
Epoch 8 Loss : 0.16278160735964775
Epoch 9 Loss : 0.1216146219521761
Epoch 10 Loss : 0.09297686070203781
Epoch 11 Loss : 0.07171512022614479
Epoch 12 Loss : 0.058209494687616825
Epoch 13 Loss : 0.044808948412537575
Epoch 14 Loss : 0.03674975037574768
Epoch 15 Loss : 0.03196391183882952
Epoch 16 Loss : 0.02686379849910736
Epoch 17 Loss : 0.022476017475128174
Epoch 18 Loss : 0.01967943413183093
Epoch 19 Loss : 0.017505895346403122
Epoch 20 Loss : 0.015201312256976962


In [ ]:
# 평가 및 예측
model.eval() # 평가모드

with torch.no_grad(): # 기울기 계산 비활성화
    output = model(X) # 순전파 (예측값 생성)
    prob = torch.sigmoid(output) # 0~1사이 확률로 변환
    pred = (prob >= 0.5).int()   # 임계값 0.5 기준으로 이진분류 (0/1)

print(labels)                            
print(pred.squeeze().detach().numpy())   # 1차원 배열형태로 변환  

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


## 사전학습된 임베딩 모델을 사용

In [17]:
from gensim.models import KeyedVectors

# 사전학습된 Word2Vec 로드
model_wv = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary = True)
model_wv.vectors.shape # (어휘 수, 300)

(3000000, 300)

In [ ]:
# 임베딩 매트릭스 초기화 후 사전학습 임베딩 차원으로 재구성
print(len(word_to_index))

# (vocab_size, embedding_dim) 크기의 0행렬 생성
embedding_matrix = np.zeros((len(word_to_index), model_wv.vectors.shape[1]))
embedding_matrix.shape

17


(17, 300)

In [ ]:
# 단어가 사전학습 모델에 있으면 임베딩 벡터 반환, 없으면 None 반환

def get_word_embedding(word):
    if word in model_wv:
        return model_wv[word]
    else:
        return None

get_word_embedding('nerd').shape    

(300,)

In [20]:
# 학습된 임베딩 벡터를 가져와 복사
for word, index in word_to_index.items():  
    if index >= 2:  # pad, oov 제외
        emb = get_word_embedding(word)
        if emb is not None:
            embedding_matrix[index] = emb # 해당 단어 인덱스 위치에 사전학습 벡터를 복사

In [ ]:
# 임베딩 매트릭스 확인 (단어별 벡터 가중치 확인)
pd.DataFrame(embedding_matrix, index = word_to_index.keys())

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
UNK,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
nice,0.158203,0.105957,-0.189453,0.386719,0.083496,-0.267578,0.083496,0.113281,-0.104004,0.178711,...,-0.085449,0.189453,-0.146484,0.134766,-0.040771,0.032715,0.089355,-0.267578,0.008362,-0.213867
great,0.071777,0.208008,-0.028442,0.178711,0.132812,-0.099609,0.096191,-0.116699,-0.008545,0.148438,...,-0.011475,0.064453,-0.289062,-0.048096,-0.199219,-0.071289,0.064453,-0.167969,-0.020874,-0.142578
best,-0.126953,0.021973,0.287109,0.153320,0.127930,0.032715,-0.115723,-0.029541,0.153320,0.011292,...,0.006439,-0.033936,-0.166016,-0.016846,-0.048584,-0.022827,-0.152344,-0.101562,-0.090332,0.088379
amazing,0.073730,0.004059,-0.135742,0.022095,0.180664,-0.046631,0.224609,-0.229492,-0.040039,0.225586,...,0.018433,-0.021240,-0.250000,-0.020142,-0.310547,-0.207031,-0.006317,-0.141602,-0.150391,-0.137695
stop,-0.057861,0.013184,0.115234,0.069824,-0.306641,-0.044678,0.048584,0.152344,0.073242,-0.100098,...,0.100098,0.171875,-0.113281,0.064453,-0.115723,0.048096,-0.004822,0.086426,0.029907,0.007812
lies,0.149414,-0.012817,0.328125,0.025513,0.017334,0.190430,0.188477,-0.143555,-0.090820,0.206055,...,-0.308594,0.183594,-0.202148,0.031494,-0.164062,-0.201172,0.080078,-0.105469,0.149414,0.157227
pitiful,0.269531,0.253906,-0.020996,0.060303,-0.010925,0.217773,0.139648,-0.057617,0.312500,0.253906,...,-0.063477,0.132812,-0.094238,0.089355,-0.065430,-0.016235,-0.107910,-0.072266,-0.094238,0.028809
nerd,0.265625,-0.207031,-0.026611,0.419922,-0.208984,0.390625,0.164062,0.063965,0.149414,-0.017700,...,0.215820,0.125000,-0.227539,-0.310547,-0.112793,-0.096680,0.255859,0.124023,-0.030273,0.082031


In [22]:
# nn.Parameter를 활용해 학습가능 파라미터를 나눔 (이론)
a = torch.tensor([1., 2., 3.], requires_grad = False)
print(a.requires_grad)
b = torch.tensor([1., 2., 3.], requires_grad = True)
print(b.requires_grad)

False
True


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 정수 시퀀스를 임베딩 -> RNN -> 선형층으로 처리해서 이진분류 LOGIT(1개) 출력하는 모델
class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix ,hidden_size = 16x):
        super().__init__()

        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,   # 단어 사전 크기
            embedding_dim = embedding_dim, # 임베딩 차원
            padding_idx = 0                # 패딩 0 인덱스는 업데이트 하지않음
        )

        # 사전학습된 임베딩 벡터로 초기화
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype = torch.float))
        # self.embedding.weight.requires_grad = False # True면 파인튜닝(추가학습), False면 임베딩 고정

        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first = True) # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환 (차후 BCEWithLogitsLoss 등 사용해서 확률값 변환해야함)
        self.out = nn.Linear(hidden_size, 1) 
    def forward(self, x):
        embedded = self.embedding(x)  # (batch, seq_len) -> (batch, seq_len, embedding_dim)
        out, h_n = self.rnn(embedded) # out:logit값 , h_n : (num_layers*directions, batch, hidden_size)
        out = self.out(h_n.squeeze(0)) # (batch, hidden_size) -> (batch,1)
        return out # logit 값

embedding_dim = model_wv.vectors.shape[] # 단어 벡터 차원 크기
model = SimpleNet(vocab_size, embedding_dim, hidden_size = 16)
model

SyntaxError: invalid decimal literal (4104838583.py, line 8)

In [1]:
# 모델학습 준비 : 텐서 변환, DataLoader 구성, 손실함수/최적화함수 설정
X = torch.tensor(padded_sequences, dtype = torch.long)     # 입력 시퀀스는 LongTensor로 변환(Embedding 입력)
y = torch.tensor(labels, dtype = torch.float).unsqueeze(1) # 라벨은 FloatTensor로 변환. (N, ) -> (N,1)

dataset = TensorDataset(X,y)
dataloader = DataLoader(dataset, batch_size = 2, shuffle = True)

criterion = nn.BCEWithLogitsLoss() # 시그모이드를 포함한 손실함수
optimizer = optim.Adam(model.parameters(), lr = 0.005)

NameError: name 'torch' is not defined

In [25]:
for epoch in range(20):
    epoch_loss = 0

    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()    # 이전 기울기 초기화
        output = model(x_batch)  # 순전파
        loss = criterion(output, y_batch) # 손실 계산
        loss.backward()          # 역전파 : 기울기 계산
        optimizer.step()         # 파라미터 업데이트

        epoch_loss += loss.item() # 미니배치 손실은 python float형태로 누적

    print(f"Epoch {epoch+1} Loss : {epoch_loss / len(dataloader)}")

Epoch 1 Loss : 0.012925280956551433
Epoch 2 Loss : 0.00841411598958075
Epoch 3 Loss : 0.00577972992323339
Epoch 4 Loss : 0.004189595929346979
Epoch 5 Loss : 0.0031593955354765058
Epoch 6 Loss : 0.002480213821399957
Epoch 7 Loss : 0.002009972231462598
Epoch 8 Loss : 0.0016829880187287927
Epoch 9 Loss : 0.001430342235835269
Epoch 10 Loss : 0.0012495368719100952
Epoch 11 Loss : 0.001108333031879738
Epoch 12 Loss : 0.0009931498061632738
Epoch 13 Loss : 0.0008981806313386187
Epoch 14 Loss : 0.000823636117274873
Epoch 15 Loss : 0.0007593387417728081
Epoch 16 Loss : 0.0007028230174910277
Epoch 17 Loss : 0.000656433156109415
Epoch 18 Loss : 0.0006132094422355294
Epoch 19 Loss : 0.0005772625008830801
Epoch 20 Loss : 0.0005432166217360646


In [26]:
# 평가 및 예측
model.eval() # 평가모드

with torch.no_grad(): # 기울기 계산 비활성화
    output = model(X) # 순전파 (예측값 생성)
    prob = torch.sigmoid(output) # 0~1사이 확률로 변환
    pred = (prob >= 0.5).int()   # 임계값 0.5 기준으로 이진분류 (0/1)

print(labels)                            
print(pred.squeeze().detach().numpy())   # 1차원 배열형태로 변환  

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]
